# LLM-SR com curvas sintéticas p-OTFT da Org4

Notebook independente para Org4: prepara a divisão por curva, inspeciona train/test_id/test_ood e controla a execução LLM-SR.

A política completa é registrada no manifesto da organização; curvas com tensões repetidas são identificadas pelo arquivo de origem.

## 1. Ambiente e configuração

Use o ambiente Conda `llmsr`. A busca real requer o servidor local LLM-SR. A célula de configuração define semente, limite de amostras e diretório de logs exclusivos desta organização.

In [ ]:
from pathlib import Path
import random
import socket
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

def find_workspace(start):
    for candidate in (start, *start.parents):
        if (candidate / "LLM-SR/main.py").is_file() and (candidate / "OTFT-SR/prepare_synthetic_org4.py").is_file():
            return candidate
    raise FileNotFoundError("Não encontrei o workspace LLM-SR/OTFT-SR.")

WORKSPACE_ROOT = find_workspace(Path.cwd().resolve())
OTFT_SR_ROOT = WORKSPACE_ROOT / "OTFT-SR"
LLM_SR_ROOT = WORKSPACE_ROOT / "LLM-SR"
sys.path.insert(0, str(LLM_SR_ROOT))
sys.path.insert(0, str(OTFT_SR_ROOT))

from prepare_synthetic_org4 import build_splits
from llmsr import config as llmsr_config
from llmsr import evaluator as llmsr_evaluator
from llmsr import pipeline as llmsr_pipeline
from llmsr import sampler as llmsr_sampler

SEED = 7
MAX_SAMPLE_NUMS = 100
RUN_PIPELINE = True
SMOKE_MODE = False
LOCAL_LLM_HOST = "127.0.0.1"
LOCAL_LLM_PORT = 5000

plt.style.use("seaborn-v0_8-whitegrid")
print("Workspace:", WORKSPACE_ROOT)

## 2. Preparação e manifesto da partição

O adaptador reconstrói os caminhos do índice para o workspace atual, seleciona curvas inteiras e amostra deterministicamente 150 pontos por curva.

In [ ]:
split_paths = build_splits()
manifest = __import__("json").loads(split_paths["manifest"].read_text(encoding="utf-8"))
split_summary = pd.DataFrame([
    {
        "split": split_name,
        "n_curvas": split_data["n_curves"],
        "n_pontos": split_data["n_points"],
        "arquivos": ", ".join(curve["csv_file"] for curve in split_data["curves"]),
    }
    for split_name, split_data in manifest["splits"].items()
])
display(split_summary)
print("Manifesto:", split_paths["manifest"])

## 3. Política científica de separação

Curvas inteiras são atribuídas a treino, validação `test_id` e extrapolação `test_ood`. O manifesto registra arquivo, tensão, família e justificativa para cada escolha. A divisão é exclusiva por curva, inclusive para tensões nominais repetidas.

In [ ]:
split_frames = {
    split_name: pd.read_csv(csv_path)
    for split_name, csv_path in split_paths.items()
    if split_name in {"train", "test_id", "test_ood"}
}
display(pd.concat({
    split_name: frame[["VGS_abs_V", "VDS_abs_V", "log10_ID_abs_A"]].agg(["count", "min", "max"]).T
    for split_name, frame in split_frames.items()
}, axis=1))

## 4. Inspeção da cobertura bivariada

As amostras usam módulos de tensão e corrente. A cor indica log10(|ID|/A), exibido em escala separada por conjunto.

In [ ]:
%matplotlib inline
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5), sharex=True, sharey=True)
for ax, (split_name, frame) in zip(axes, split_frames.items()):
    points = ax.scatter(
        frame["VGS_abs_V"], frame["VDS_abs_V"],
        c=frame["log10_ID_abs_A"], s=9, alpha=.65, cmap="viridis"
    )
    ax.set(title=split_name, xlabel="|VGS| (V)", ylabel="|VDS| (V)")
    fig.colorbar(points, ax=ax, label="log10(|ID|/A)")
plt.suptitle("Cobertura de polarização por curva reservada")
plt.tight_layout()
plt.show()

## 5. Especificação e objeto de dados

Somente o conjunto train é entregue ao otimizador. test_id e test_ood permanecem reservados para avaliação fora da busca.

In [ ]:
train_frame = split_frames["train"]
input_columns = ["VGS_abs_V", "VDS_abs_V"]
X_train = train_frame[input_columns].to_numpy(dtype=np.float64)
y_train = train_frame["log10_ID_abs_A"].to_numpy(dtype=np.float64)
pipeline_dataset = {"data": {"inputs": X_train, "outputs": y_train}}
specification_path = OTFT_SR_ROOT / "specifications/otft_synthetic_org4_numpy.txt"
specification = specification_path.read_text(encoding="utf-8")
RUN_DIRECTORY = OTFT_SR_ROOT / "logs/org4_synthetic_notebook"
print("X_train:", X_train.shape, "| y_train:", y_train.shape)
print("Especificação:", specification_path)

## 6. Execução opcional do pipeline

RUN_PIPELINE=True inicia a busca configurada. Para execução real, o servidor de completions local precisa estar ativo. SMOKE_MODE=True substitui apenas o gerador de candidatas por uma hipótese determinística para verificar o encadeamento sem servidor.

In [ ]:
if RUN_PIPELINE and not SMOKE_MODE:
    try:
        with socket.create_connection((LOCAL_LLM_HOST, LOCAL_LLM_PORT), timeout=2):
            print("Servidor local LLM disponível.")
    except OSError as error:
        raise RuntimeError("Inicie o servidor LLM-SR antes de executar a busca.") from error

if RUN_PIPELINE:
    random.seed(SEED)
    np.random.seed(SEED)
    llmsr_sampler.Sampler._global_samples_nums = 1

    if SMOKE_MODE:
        class DeterministicSmokeLLM(llmsr_sampler.LLM):
            def draw_samples(self, prompt, config=None):
                del prompt, config
                return [
                    "    return parameters[0] + parameters[1] * gate_voltage_abs + "
                    "parameters[2] * drain_voltage_abs + "
                    "parameters[3] * gate_voltage_abs * drain_voltage_abs"
                ]
        llm_class = DeterministicSmokeLLM
    else:
        llm_class = llmsr_sampler.LocalLLM

    class_config = llmsr_config.ClassConfig(
        llm_class=llm_class,
        sandbox_class=llmsr_evaluator.LocalSandbox,
    )
    experiment_config = llmsr_config.Config(
        problem_name="org4_otft_synthetic",
        samples_per_prompt=1 if SMOKE_MODE else 4,
    )
    RUN_DIRECTORY.mkdir(parents=True, exist_ok=True)
    llmsr_pipeline.main(
        specification=specification,
        inputs=pipeline_dataset,
        config=experiment_config,
        max_sample_nums=MAX_SAMPLE_NUMS,
        class_config=class_config,
        log_dir=str(RUN_DIRECTORY),
    )
else:
    print("Busca desativada; conjunto de treino foi preparado e inspecionado.")

## 7. Artefatos e avaliação

A divisão exata está em curve_split_manifest.json; os CSVs gerados são reutilizados pelo runner modular. Scores da busca medem MSE somente em treino. Para avaliar generalização, aplique a expressão selecionada a test_id e test_ood sem reajustar parâmetros.

In [ ]:
print("Arquivos da partição:")
for split_name in ["train", "test_id", "test_ood", "manifest"]:
    print(split_name, "->", split_paths[split_name])